In [1]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 🔹 10.8.1 — Load Full EPC File (Preserve All Columns)
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# 🔸 Filter for buildings in Milan only
epc_milan = epc_df[epc_df["COMUNE"].str.upper() == "MILANO"].copy()
print(f"Filtered EPC for Milan → Shape: {epc_milan.shape}")

# 🔹 10.8.2 — Convert to GeoDataFrame using coordinates
epc_milan["geometry"] = [Point(xy) for xy in zip(epc_milan["Z"], epc_milan["Y"])]
epc_gdf = gpd.GeoDataFrame(epc_milan, geometry="geometry", crs="EPSG:4326")

# 🔹 10.8.3 — Load Buildings + Climate (already with geometry + EPSG:4326)
bldg_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"
bldg_gdf = gpd.read_file(bldg_path)

# 🔹 10.8.4 — Nearest Spatial Join (each EPC matched to closest building)
joined = gpd.sjoin_nearest(epc_gdf, bldg_gdf, how="left", distance_col="dist_to_building")

# 🧪 Optional: check join success
matched_pct = 100 * joined["building_id"].notnull().mean()
print(f"Successfully matched {matched_pct:.2f}% of EPC entries to buildings.")

# 🔹 10.8.5 — Save Final Result (all EPC columns + matched building info)
output_csv = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\epc_matched_to_buildings.csv"
output_gpkg = output_csv.replace(".csv", ".geojson")

joined.to_csv(output_csv, index=False)
joined.to_file(output_gpkg, driver="GeoJSON")

print("✅ Saved:")
print("→", output_csv)
print("→", output_gpkg)


Filtered EPC for Milan → Shape: (342684, 198)


ValueError: could not convert string to float: np.str_('C (64,47)')

In [2]:
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point

# 🔹 10.8.1 — Load Full EPC File (Preserve All Columns)
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# 🔸 Filter for Milan only (case-insensitive)
epc_milan = epc_df[epc_df["COMUNE"].str.upper() == "MILANO"].copy()
print(f"Filtered EPC for Milan → Shape: {epc_milan.shape}")

# 🔹 10.8.2 — Clean Coordinates and Convert to GeoDataFrame
# Safely convert Y (latitude) and Z (longitude) to floats
epc_milan["Y"] = pd.to_numeric(epc_milan["Y"], errors="coerce")
epc_milan["Z"] = pd.to_numeric(epc_milan["Z"], errors="coerce")

# Drop rows with invalid or missing coordinates
before = epc_milan.shape[0]
epc_milan = epc_milan.dropna(subset=["Y", "Z"])
after = epc_milan.shape[0]
print(f"Dropped {before - after} rows with invalid coordinates.")

# Construct geometry
epc_milan["geometry"] = [Point(xy) for xy in zip(epc_milan["Z"], epc_milan["Y"])]
epc_gdf = gpd.GeoDataFrame(epc_milan, geometry="geometry", crs="EPSG:4326")

# 🔹 10.8.3 — Load Buildings with Climate Info
bldg_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"
bldg_gdf = gpd.read_file(bldg_path)

# 🔹 10.8.4 — Perform Nearest Spatial Join
joined = gpd.sjoin_nearest(epc_gdf, bldg_gdf, how="left", distance_col="dist_to_building")

# 🧪 Check join success rate
matched_pct = 100 * joined["building_id"].notnull().mean()
print(f"✅ Matched {matched_pct:.2f}% of EPC entries to buildings.")

# 🔹 10.8.5 — Save Final Dataset (CSV and GeoJSON)
output_csv = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\epc_matched_to_buildings.csv"
output_gpkg = output_csv.replace(".csv", ".geojson")

joined.to_csv(output_csv, index=False)
joined.to_file(output_gpkg, driver="GeoJSON")

print("🎯 Saved outputs:")
print("→", output_csv)
print("→", output_gpkg)


Filtered EPC for Milan → Shape: (342684, 198)
Dropped 342684 rows with invalid coordinates.


F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\geopandas\array.py:408: UserWarning: Geometry is in a geographic CRS. Results from 'sjoin_nearest' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  warnings.warn(


✅ Matched nan% of EPC entries to buildings.
🎯 Saved outputs:
→ F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\epc_matched_to_buildings.csv
→ F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\epc_matched_to_buildings.geojson


In [3]:
# Check what these columns actually contain (non-null samples)
print("Sample Y:", epc_df["Y"].dropna().head(10).tolist())
print("Sample Z:", epc_df["Z"].dropna().head(10).tolist())

# Also: check data types
print("Dtypes:")
print(epc_df[["Y", "Z"]].dtypes)


Sample Y: ['C (64,47)', 'B (99,4)', 'B (42,87)', 'B (82,17)', 'A1 (72,45)', 'A3 (20,01)', 'C (110,01)', 'B (85,89)', 'B (62,43)', 'B (42,38)']
Sample Z: []
Dtypes:
Y     object
Z    float64
dtype: object


In [4]:
import pandas as pd

# Load a sample (first 1000 rows) from the original file
original_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\bbky-sde5_version_17245.csv"
df_original = pd.read_csv(original_path, sep=",", nrows=1000)

# Check shape and column names
print(f"Original File Shape: {df_original.shape}")
print("Sample columns:", df_original.columns[:20])

# Show coordinate columns
coord_cols = ['wgs84_x', 'wgs84_y', 'location', 'Y', 'Z']
for col in coord_cols:
    if col in df_original.columns:
        print(f"\n==> Column: {col} (type: {df_original[col].dtype})")
        print(df_original[col].dropna().head(10))
    else:
        print(f"\nColumn '{col}' not found in dataset.")


Original File Shape: (1000, 210)
Sample columns: Index(['ci_anno_installazione_2', 'tpc_anno_installazione_2',
       'tpc_anno_installazione_1', 'ill_anno_installazione',
       'pfr_anno_installazione_2', 'pfr_anno_installazione_1',
       'ic_anno_installazione', 'pa_anno_installazione',
       'ci_anno_installazione_1', 'ce_anno_installazione_2',
       'ce_anno_installazione_1', 'sez_censimento', 'cod_ape', 'data_ins',
       'residenziale', 'non_residenziale', 'comune_catastale', 'sezione',
       'foglio', 'particella'],
      dtype='object')

==> Column: wgs84_x (type: float64)
Series([], Name: wgs84_x, dtype: float64)

==> Column: wgs84_y (type: float64)
Series([], Name: wgs84_y, dtype: float64)

==> Column: location (type: float64)
Series([], Name: location, dtype: float64)

Column 'Y' not found in dataset.

Column 'Z' not found in dataset.


In [5]:
import pandas as pd

# Load the cleaned EPC dataset
epc_path = 'F:/Building Engineering - Polytechnic University of Turin/Masters Thesis/23-11-2025/Data Collections/CENED/Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv'
epc_df = pd.read_csv(epc_path)

# Drop rows where Y is missing or not a string with parentheses
epc_df = epc_df[epc_df['Y'].astype(str).str.contains(r'\([\d,\.]+\)', na=False)]

# Extract numeric value inside parentheses from 'Y' column
epc_df['coord_y'] = (
    epc_df['Y']
    .astype(str)
    .str.extract(r'\(([\d,\.]+)\)', expand=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# Show result and confirm
print("New coord_y extracted:")
print(epc_df[['Y', 'coord_y']].head())

# Optional: save intermediate version
epc_df.to_csv('epc_with_coord_y.csv', index=False)


C:\Users\Nima\AppData\Local\Temp\ipykernel_10384\3906543911.py:5: DtypeWarning: Columns (112,175,182,192,193,194) have mixed types. Specify dtype option on import or set low_memory=False.
  epc_df = pd.read_csv(epc_path)


New coord_y extracted:
            Y  coord_y
0   C (64,47)    64.47
1    B (99,4)    99.40
2   B (42,87)    42.87
3   B (82,17)    82.17
4  A1 (72,45)    72.45


In [6]:
import pandas as pd

# Paths
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_address_reference.csv"

# Load EPC data (cleaned)
epc = pd.read_csv(epc_path, low_memory=False)
epc = epc[epc["COMUNE"].str.upper() == "MILANO"].copy()

# Normalize EPC address
epc["INDIRIZZO_NORM"] = epc["INDIRIZZO"].astype(str).str.upper().str.strip().str.replace("VIALE", "VIA").str.replace("V.", "VIA").str.replace("  ", " ")

# Load buildings reference (already processed to contain clean 'address_clean')
buildings = pd.read_csv(buildings_path)

# Preview
print("EPC sample addresses:", epc["INDIRIZZO_NORM"].dropna().unique()[:5])
print("Building reference addresses:", buildings["address_clean"].dropna().unique()[:5])


EPC sample addresses: ['FONTANELLI 10' 'VIA CALDIROLA 6' 'VIA G. SILVA 43' 'VIA MAHON 119'
 'VIA TUNISIA 42']


KeyError: 'address_clean'

In [7]:
# Show all available columns in the buildings reference file
print("Available columns in buildings reference:")
print(buildings.columns.tolist())

# Also show sample rows to visually inspect address fields
print("\nSample rows from buildings:")
print(buildings.head(3))


Available columns in buildings reference:
['building_id', 'centroid_lon', 'centroid_lat', 'INDIRIZZO_NORM']

Sample rows from buildings:
    building_id  centroid_lon  centroid_lat  INDIRIZZO_NORM
0  E00000000001      9.223222     45.496484               1
1  E00000000002      9.234857     45.493874               2
2  E00000000003      9.220783     45.495266               3


In [8]:
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Load EPC dataset
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# Step 1: Filter to MILANO only
epc_df = epc_df[epc_df["COMUNE"].str.upper() == "MILANO"].copy()

# Step 2: Create full address field
epc_df["full_address"] = epc_df["INDIRIZZO"].astype(str).str.strip() + ", MILANO, ITALY"

# Step 3: Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="epc_geocoder_milano")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Step 4: Geocode addresses (limit to first 100 for testing)
results = []
for address in epc_df["full_address"].unique()[:100]:  # Remove the [:100] slice later to process all
    try:
        location = geocode(address)
        if location:
            results.append({"full_address": address, "latitude": location.latitude, "longitude": location.longitude})
        else:
            results.append({"full_address": address, "latitude": None, "longitude": None})
    except Exception as e:
        results.append({"full_address": address, "latitude": None, "longitude": None})
        print(f"Error for {address}: {e}")

# Step 5: Create lookup DataFrame and merge back
geo_df = pd.DataFrame(results)
epc_geo = epc_df.merge(geo_df, on="full_address", how="left")

# Step 6: Save for future stages
epc_geo.to_csv("epc_geocoded_milano.csv", index=False)

# Preview
print("Geocoded sample:")
print(epc_geo[["full_address", "latitude", "longitude"]].dropna().head())


ModuleNotFoundError: No module named 'geopy'

In [9]:
!pip install geopy


  Obtaining dependency information for geopy from https://files.pythonhosted.org/packages/e5/15/cf2a69ade4b194aa524ac75112d5caac37414b20a3a03e6865dfe0bd1539/geopy-2.4.1-py3-none-any.whl.metadata
  Obtaining dependency information for geographiclib<3,>=1.52 from https://files.pythonhosted.org/packages/31/b3/802576f2ea5dcb48501bb162e4c7b7b3ca5654a42b2c968ef98a797a4c79/geographiclib-2.1-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   ---------------------------------------- 0.0/125.4 kB ? eta -:--:--
   -------------------------- ------------- 81.9/125.4 kB 2.3 MB/s eta 0:00:01
   ----------------------------------- ---- 112.6/125.4 kB 1.3 MB/s eta 0:00:01
   -------------------------------------- 125.4/125.4 kB 921.6 kB/s eta 0:00:00
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------------------------------------- 0.0/40.7 kB ? eta -:--:--
   ---------- ----------------------------- 10.2/40.7 kB ? eta -:-


[notice] A new release of pip is available: 23.2.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Load EPC dataset
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# Step 1: Filter to MILANO only
epc_df = epc_df[epc_df["COMUNE"].str.upper() == "MILANO"].copy()

# Step 2: Create full address field
epc_df["full_address"] = epc_df["INDIRIZZO"].astype(str).str.strip() + ", MILANO, ITALY"

# Step 3: Initialize Nominatim geocoder
geolocator = Nominatim(user_agent="epc_geocoder_milano")
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1)

# Step 4: Geocode addresses (limit to first 100 for testing)
results = []
for address in epc_df["full_address"].unique()[:100]:  # Remove the [:100] slice later to process all
    try:
        location = geocode(address)
        if location:
            results.append({"full_address": address, "latitude": location.latitude, "longitude": location.longitude})
        else:
            results.append({"full_address": address, "latitude": None, "longitude": None})
    except Exception as e:
        results.append({"full_address": address, "latitude": None, "longitude": None})
        print(f"Error for {address}: {e}")

# Step 5: Create lookup DataFrame and merge back
geo_df = pd.DataFrame(results)
epc_geo = epc_df.merge(geo_df, on="full_address", how="left")

# Step 6: Save for future stages
epc_geo.to_csv("epc_geocoded_milano.csv", index=False)

# Preview
print("Geocoded sample:")
print(epc_geo[["full_address", "latitude", "longitude"]].dropna().head())


RateLimiter caught an error, retrying (0/2 tries). Called with (*('Corso Garibaldi 46, MILANO, ITALY',), **{}).
Traceback (most recent call last):
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connectionpool.py", line 534, in _make_request
    response = conn.getresponse()
               ^^^^^^^^^^^^^^^^^^
  File "F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\venv\Lib\site-packages\urllib3\connection.py", line 565, in getresponse
    httplib_response = super().getresponse()
                       ^^^^^^^^^^^^^^^^^^^^^
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 1374, in getresponse
    response.begin()
  File "G:\P R O G R A M S\Python\Lib\http\client.py", line 318, in begin
    version, status, reason = self._read_status()
                              ^^^^^^^^^^^^^^^^^^^
  Fil

Geocoded sample:
                      full_address   latitude  longitude
0     fontanelli 10, MILANO, ITALY  45.525040   9.174245
1   Via Caldirola 6, MILANO, ITALY  45.513779   9.213327
3     VIA MAHON 119, MILANO, ITALY  45.498541   9.151539
4  Viale Tunisia 42, MILANO, ITALY  45.479438   9.201112
5     Via Rimini 30, MILANO, ITALY  45.440872   9.170377


In [11]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Safer geocoder: longer timeout and more retries
geolocator = Nominatim(user_agent="epc_geocoder_milano", timeout=10)  # timeout = 10 seconds
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=5)


In [12]:
import pandas as pd
import time
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# ----------------------------------------------------------
# 1. Load the EPC dataset
epc_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\Data Collections\CENED\Database_CENED+2_-_Certificazione_ENergetica_degli_EDifici_20251128.csv"
epc_df = pd.read_csv(epc_path, low_memory=False)

# ----------------------------------------------------------
# 2. Filter only for buildings in MILANO
epc_df = epc_df[epc_df["COMUNE"].str.upper() == "MILANO"].copy()

# ----------------------------------------------------------
# 3. Build full address string (street + city + country)
epc_df["INDIRIZZO"] = epc_df["INDIRIZZO"].astype(str).str.strip()
epc_df["full_address"] = epc_df["INDIRIZZO"] + ", MILANO, ITALY"

# ----------------------------------------------------------
# 4. Set up geocoder (with increased timeout + retries)
geolocator = Nominatim(user_agent="epc_geocoder_milano", timeout=10)
geocode = RateLimiter(geolocator.geocode, min_delay_seconds=1, max_retries=3, error_wait_seconds=5)

# ----------------------------------------------------------
# 5. Geocode a test sample of addresses (e.g., first 25 unique)
sample_addresses = epc_df["full_address"].dropna().unique()[:25]

geo_results = []
for addr in sample_addresses:
    try:
        location = geocode(addr)
        if location:
            geo_results.append({"full_address": addr, "latitude": location.latitude, "longitude": location.longitude})
        else:
            geo_results.append({"full_address": addr, "latitude": None, "longitude": None})
    except Exception as e:
        geo_results.append({"full_address": addr, "latitude": None, "longitude": None})
        print(f"❌ Error for address '{addr}': {e}")

# ----------------------------------------------------------
# 6. Merge coordinates back to EPC dataframe
geo_df = pd.DataFrame(geo_results)
epc_geo = epc_df.merge(geo_df, on="full_address", how="left")

# ----------------------------------------------------------
# 7. Save the geocoded sample
epc_geo.to_csv("epc_geocoded_milano.csv", index=False)

# ----------------------------------------------------------
# 8. Preview result
print("✅ Geocoded sample:")
print(epc_geo[["full_address", "latitude", "longitude"]].dropna().head())


✅ Geocoded sample:
                      full_address   latitude  longitude
0     fontanelli 10, MILANO, ITALY  45.525040   9.174245
1   Via Caldirola 6, MILANO, ITALY  45.513779   9.213327
3     VIA MAHON 119, MILANO, ITALY  45.498541   9.151539
4  Viale Tunisia 42, MILANO, ITALY  45.479438   9.201112
5     Via Rimini 30, MILANO, ITALY  45.440872   9.170377


In [13]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point

# === 1. Load geocoded EPC sample ===
epc_sample_path = "epc_geocoded_milano.csv"  # make sure this path is correct
epc_df = pd.read_csv(epc_sample_path)

# Drop rows with missing coordinates (just in case)
epc_df = epc_df.dropna(subset=["latitude", "longitude"])

# === 2. Convert to GeoDataFrame ===
geometry = [Point(xy) for xy in zip(epc_df["longitude"], epc_df["latitude"])]
epc_gdf = gpd.GeoDataFrame(epc_df, geometry=geometry, crs="EPSG:4326")

# === 3. Preview and save ===
print("EPC GeoDataFrame sample:")
print(epc_gdf[["full_address", "geometry"]].head())

# Optional: save to GeoJSON for later steps
output_path = "epc_geocoded_sample.geojson"
epc_gdf.to_file(output_path, driver="GeoJSON")
print(f"Saved to: {output_path}")


C:\Users\Nima\AppData\Local\Temp\ipykernel_10384\1813477358.py:7: DtypeWarning: Columns (112,175,182,192,193,194) have mixed types. Specify dtype option on import or set low_memory=False.
  epc_df = pd.read_csv(epc_sample_path)


EPC GeoDataFrame sample:
                      full_address                  geometry
0     fontanelli 10, MILANO, ITALY  POINT (9.17424 45.52504)
1   Via Caldirola 6, MILANO, ITALY  POINT (9.21333 45.51378)
3     VIA MAHON 119, MILANO, ITALY  POINT (9.15154 45.49854)
4  Viale Tunisia 42, MILANO, ITALY  POINT (9.20111 45.47944)
5     Via Rimini 30, MILANO, ITALY  POINT (9.17038 45.44087)
Saved to: epc_geocoded_sample.geojson


In [14]:
import geopandas as gpd

# === 1. Load spatial EPC sample and building polygons ===
epc_gdf = gpd.read_file("epc_geocoded_sample.geojson")
buildings_gdf = gpd.read_file("buildings_with_climate.geojson")

# Ensure both are in same CRS
epc_gdf = epc_gdf.to_crs("EPSG:4326")
buildings_gdf = buildings_gdf.to_crs("EPSG:4326")

# === 2. Spatial join — assign EPCs to buildings ===
epc_with_building = gpd.sjoin(epc_gdf, buildings_gdf[["building_id", "geometry"]], how="left", predicate="within")

# === 3. Check result ===
print("Spatial join result:")
print(epc_with_building[["full_address", "building_id"]].head())

# Count how many EPCs were successfully matched to buildings
matched = epc_with_building["building_id"].notna().sum()
total = len(epc_with_building)
print(f"✅ Matched {matched} out of {total} EPC entries to buildings.")

# === 4. Save for next step ===
epc_with_building.to_file("epc_with_building_id.geojson", driver="GeoJSON")
print("Saved to: epc_with_building_id.geojson")


DataSourceError: buildings_with_climate.geojson: No such file or directory

In [15]:
import geopandas as gpd

# === 1. Load spatial EPC sample and building polygons ===
epc_gdf = gpd.read_file("epc_geocoded_sample.geojson")

# Full path to buildings_with_climate.geojson
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"
buildings_gdf = gpd.read_file(buildings_path)

# === 2. Ensure both GeoDataFrames use the same CRS ===
epc_gdf = epc_gdf.to_crs("EPSG:4326")
buildings_gdf = buildings_gdf.to_crs("EPSG:4326")

# === 3. Perform spatial join (point-in-polygon) ===
epc_with_building = gpd.sjoin(
    epc_gdf,
    buildings_gdf[["building_id", "geometry"]],
    how="left",
    predicate="within"
)

# === 4. Check result ===
print("Spatial join result (first 5 rows):")
print(epc_with_building[["full_address", "building_id"]].head())

matched = epc_with_building["building_id"].notna().sum()
total = len(epc_with_building)
print(f"✅ Matched {matched} of {total} EPC entries to buildings.")

# === 5. Save result for later integration ===
output_path = "epc_with_building_id.geojson"
epc_with_building.to_file(output_path, driver="GeoJSON")
print(f"Saved joined EPC data to: {output_path}")


Spatial join result (first 5 rows):
                      full_address   building_id
0     fontanelli 10, MILANO, ITALY           NaN
1   Via Caldirola 6, MILANO, ITALY  E00000039761
2     VIA MAHON 119, MILANO, ITALY           NaN
3  Viale Tunisia 42, MILANO, ITALY  E00000024968
4     Via Rimini 30, MILANO, ITALY           NaN
✅ Matched 46 of 157 EPC entries to buildings.
Saved joined EPC data to: epc_with_building_id.geojson


In [16]:
import geopandas as gpd
import pandas as pd

# === 1. Load EPC data with building IDs ===
epc_path = "epc_with_building_id.geojson"
epc_matched = gpd.read_file(epc_path)

# Drop rows where building_id is missing
epc_matched = epc_matched[epc_matched["building_id"].notna()].copy()

# === 2. Load full building + climate dataset ===
buildings_path = r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate.geojson"
buildings_gdf = gpd.read_file(buildings_path)

# === 3. Merge on building_id (inner join to keep matched buildings only) ===
ml_df = pd.merge(
    epc_matched.drop(columns="geometry"),  # drop point geometry
    buildings_gdf.drop(columns="geometry"),  # drop polygon geometry
    on="building_id",
    how="inner"
)

# === 4. Export to CSV ===
output_csv = "ml_sample_buildings.csv"
ml_df.to_csv(output_csv, index=False)
print(f"✅ ML-ready sample saved to: {output_csv}")
print(f"Shape: {ml_df.shape}")


✅ ML-ready sample saved to: ml_sample_buildings.csv
Shape: (46, 225)


In [1]:
import os

# Define the full paths of files you want to delete
files_to_delete = [
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_geocoded_sample.geojson",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_with_building_id.geojson",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_with_coord_y.csv",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\ml_sample_buildings.csv",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_geocoded_milano.xlsx",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_ready_for_epc.geojson",
    r"F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_with_climate_2d.geojson",
]

# Loop and delete each file if it exists
for file_path in files_to_delete:
    try:
        if os.path.exists(file_path):
            os.remove(file_path)
            print(f"✅ Deleted: {file_path}")
        else:
            print(f"⚠️ File not found (skipped): {file_path}")
    except Exception as e:
        print(f"❌ Error deleting {file_path}: {e}")


✅ Deleted: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_geocoded_sample.geojson
✅ Deleted: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_with_building_id.geojson
✅ Deleted: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_with_coord_y.csv
⚠️ File not found (skipped): F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\ml_sample_buildings.csv
⚠️ File not found (skipped): F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\epc_geocoded_milano.xlsx
✅ Deleted: F:\Building Engineering - Polytechnic University of Turin\Masters Thesis\23-11-2025\PythonProjects\ClimateProcessingProject\data\Buildings\buildings_re